## LLM Structured Output

<a href="https://colab.research.google.com/github/HassanAlgoz/dl/blob/main/modules/pipelines/02-pipelines/05_llm_structured_output.ipynb" target="_blank">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"/>
</a>

_Click the badge above to open and run this notebook in Google Colab!_

## Overview

LLMs are powerful but their outputs are unpredictable. Most solutions attempt to fix bad outputs after generation using parsing, regex, or fragile code that breaks easily.

[Outlines](https://dottxt-ai.github.io/outlines/latest/) guarantees structured outputs during generation — directly from any LLM.

- **Works with any model** - Same code runs across OpenAI, Ollama, vLLM, and more
- **Simple integration** - Just pass your desired output type: `model(prompt, output_type)`
- **Guaranteed valid structure** - No more parsing headaches or broken JSON
- **Provider independence** - Switch models without changing code
- **Rich structure definition** - Use Json Schema, regular expressions or context-free grammars


![](https://github.com/dottxt-ai/outlines/raw/main/docs/assets/images/logo-light-mode.svg#gh-light-mode-only){height=100}


## Set up

In [1]:
!pip install -qqq torch
!pip install -Uqqq transformers datasets evaluate accelerate timm

### Suppress output logs

In [2]:
import os
import logging

from huggingface_hub.utils import disable_progress_bars
from transformers.utils import logging as transformers_logging

os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
disable_progress_bars()
transformers_logging.set_verbosity_error()
transformers_logging.disable_progress_bar()
logging.getLogger("huggingface_hub").setLevel(logging.ERROR)

In [3]:
! pip install -qqq outlines

## Connect with Transformers

In [4]:
import outlines
from transformers import AutoModelForCausalLM, AutoTokenizer

# Define the model you want to use
model_name = "Qwen/Qwen2.5-0.5B-Instruct"

# Create a HuggingFace model and tokenizer
hf_model = AutoModelForCausalLM.from_pretrained(model_name)
hf_tokenizer = AutoTokenizer.from_pretrained(model_name)

# Create an Outlines model
model = outlines.from_transformers(hf_model, hf_tokenizer)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:134: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


### Start with simple structured outputs

In [5]:
# Extract specific types
temp = model("What's the boiling point of water in Celsius?", int, max_new_tokens=5)
print(temp)

10000


### Multiple-choice

In [6]:
from typing import Literal

# Simple classification
sentiment = model(
    "Analyze: 'This product completely changed my life!'",
    Literal["Positive", "Negative", "Neutral"]
)
print(sentiment)  # "Positive"

/usr/local/lib/python3.12/dist-packages/transformers/generation/utils.py:1616: UserWarning: Using the model-agnostic default `max_length` (=60) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(


Negative


### Create complex structures

In [7]:
from pydantic import BaseModel
from enum import Enum

class Rating(Enum):
    poor = 1
    fair = 2
    good = 3
    excellent = 4

class ProductReview(BaseModel):
    rating: Rating
    pros: list[str]
    cons: list[str]
    summary: str

review = model(
    "Review: The XPS 13 has great battery life and a stunning display, but it runs hot and the webcam is poor quality.",
    ProductReview,
    max_new_tokens=200,
)

review = ProductReview.model_validate_json(review)
print(f"Rating: {review.rating.name}")  # "Rating: good"
print(f"Pros: {review.pros}")           # "Pros: ['great battery life', 'stunning display']"
print(f"Summary: {review.summary}")     # "Summary: Good laptop with great display but thermal issues"

Rating: fair
Pros: ['Great battery life', 'Stunning display']
Summary: The XPS 13 is a high-end laptop with impressive specs that offers both good performance and decent features. However, its overall experience can be quite unpleasant due to overheating and subpar camera quality.
